# ANÁLISE DE VENDAS DE ECOMMERCE

## Introdução

## Preparação

In [1]:
# importação das bibliotecas necessárias
import pandas as pd
import sqlite3


In [2]:
#ler o arquivo
df = pd.read_csv('vendas_desafio.csv')

# conferir a integridade dos dados
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id_venda        10000 non-null  int64  
 1   data_venda      10000 non-null  str    
 2   cliente         10000 non-null  str    
 3   produto         10000 non-null  str    
 4   categoria       10000 non-null  str    
 5   quantidade      10000 non-null  int64  
 6   preco_unitario  10000 non-null  float64
 7   vendedor        10000 non-null  str    
 8   cidade          10000 non-null  str    
 9   estado          10000 non-null  str    
dtypes: float64(1), int64(2), str(7)
memory usage: 781.4 KB


In [ ]:
# criar conexão sqlite3
conn = sqlite3.connect('analise_vendas.db')

# criar o banco de dados a partir do csv
df.to_sql(
    'vendas', # nome da tabela
    con = conn, # conexão criada acima
    if_exists='replace', # se já existir a tabela, apaga e cria outra
    index=False # não carregar os índices do dataframe na tabela
)

# não esquecer de fechar a conexão
conn.close()

A partir desse ponto já foi criado o banco de dados .db com os dados do .csv original. Assim, já é possível usar comando SQL para iniciar as atividades propostas.

In [25]:
df.nunique()

id_venda          10000
data_venda          365
cliente             282
produto              10
categoria             4
quantidade            4
preco_unitario     8751
vendedor              5
cidade                5
estado                5
dtype: int64

In [4]:
df.head(15)

,id_venda,data_venda,cliente,produto,categoria,quantidade,preco_unitario,vendedor,cidade,estado
0,1,2024-09-27,Pietra,Mouse,Eletrônicos,3,84.74,Ana,Belo Horizonte,RJ
1,2,2024-01-21,Vinicius,Monitor,Eletrônicos,3,950.34,Bruno,São Paulo,RS
2,3,2024-08-02,Yago,Mouse,Eletrônicos,3,79.13,Eduardo,Curitiba,SP
3,4,2024-04-09,Lara,Notebook,Eletrônicos,3,3471.47,Ana,Rio de Janeiro,RJ
4,5,2024-05-29,Levi,Fone,Eletrônicos,4,208.32,Eduardo,São Paulo,RS
5,6,2024-10-20,Ravi Lucca,Monitor,Eletrônicos,1,984.58,Eduardo,Curitiba,RJ
6,7,2024-10-03,Thales,Livro SQL,Livros,2,93.64,Eduardo,Belo Horizonte,SP
7,8,2024-01-22,José,Teclado,Eletrônicos,1,144.13,Daniela,Belo Horizonte,MG
8,9,2024-02-18,Isabel,Teclado,Eletrônicos,1,147.96,Bruno,Belo Horizonte,SP
9,10,2024-06-18,Luan,Mouse,Eletrônicos,3,81.79,Daniela,São Paulo,MG


## Atividades em SQL

### 1. Qual o faturamento total por produto?

In [5]:
# abre a conexão com o banco de dados
conn = sqlite3.connect("analise_vendas.db")

# a query a ser executada
query = """ 
    SELECT
        produto,
        'R$ ' || format('%,.2f', SUM(quantidade * preco_unitario)) AS faturamento_total
    FROM vendas 
    GROUP BY produto
    ORDER BY SUM(quantidade * preco_unitario) DESC
"""

# execução da query criada acima
df_resposta = pd.read_sql_query(query, con=conn)

# não esquecer de fechar a conexão
conn.close()

# exibir a resposta
df_resposta

,produto,faturamento_total
0,Notebook,"R$ 8,290,852.39"
1,Mesa,"R$ 3,092,939.21"
2,Monitor,"R$ 2,364,641.86"
3,Cadeira,"R$ 1,629,832.30"
4,Fone,"R$ 495,633.09"
5,Mochila,"R$ 460,376.91"
6,Teclado,"R$ 384,771.59"
7,Livro Python,"R$ 296,630.22"
8,Livro SQL,"R$ 247,460.70"
9,Mouse,"R$ 205,764.16"


_nota:_ A concatenação de 'R$' e o uso de _format_ são apenas para facilitar a visualização do resultado. O número em si dentro do banco de dados continua como um float.

### 2. Qual o faturamento total por categoria?

In [6]:
# abre a conexão com o banco de dados
conn = sqlite3.connect("analise_vendas.db")

# a query a ser executada
query = """ 
    SELECT
        categoria,
        'R$ ' || format('%,.2f', SUM(quantidade * preco_unitario)) AS faturamento_total
    FROM vendas 
    GROUP BY categoria
    ORDER BY SUM(quantidade * preco_unitario) DESC
"""

# execução da query criada acima
df_resposta = pd.read_sql_query(query, con=conn)

# não esquecer de fechar a conexão
conn.close()

# exibir a resposta
df_resposta

,categoria,faturamento_total
0,Eletrônicos,"R$ 11,741,663.09"
1,Móveis,"R$ 4,722,771.51"
2,Livros,"R$ 544,090.92"
3,Acessórios,"R$ 460,376.91"


### 3. Qual o ticket médio por cliente?

In [7]:
# abre a conexão com o banco de dados
conn = sqlite3.connect("analise_vendas.db")

# a query a ser executada
query = """ 
    SELECT
        cliente,
        (SUM(quantidade * preco_unitario) / COUNT(cliente)) AS ticket_medio
    FROM vendas 
    GROUP BY cliente
    ORDER BY ticket_medio DESC
"""

# execução da query criada acima
df_resposta = pd.read_sql_query(query, con=conn)

# não esquecer de fechar a conexão
conn.close()

# exibir a resposta
df_resposta #TODO: Arrumar a visualização de moeda

,cliente,ticket_medio
0,Mathias,2991.260732
1,Isabelly,2969.923704
2,Thiago,2828.269474
3,Danilo,2814.017500
4,Nicole,2767.899355
...,...,...
277,Maria Liz,756.460000
278,Liam,730.245641
279,Gabriel,686.122564
280,Mariane,653.825294


### 4. Qual o faturamento total por vendedor?

In [8]:
# abre a conexão com o banco de dados
conn = sqlite3.connect("analise_vendas.db")

# a query a ser executada
query = """ 
    SELECT
        vendedor,
        'R$ ' || format('%,.2f', SUM(quantidade * preco_unitario)) AS faturamento_total
    FROM vendas 
    GROUP BY vendedor
    ORDER BY SUM(quantidade * preco_unitario) DESC
"""

# execução da query criada acima
df_resposta = pd.read_sql_query(query, con=conn)

# não esquecer de fechar a conexão
conn.close()

# exibir a resposta
df_resposta

,vendedor,faturamento_total
0,Ana,"R$ 3,544,588.45"
1,Bruno,"R$ 3,501,930.96"
2,Eduardo,"R$ 3,477,302.18"
3,Carlos,"R$ 3,476,724.22"
4,Daniela,"R$ 3,468,356.62"


### 5. Qual o faturamento por mês?

In [21]:
# abre a conexão com o banco de dados
conn = sqlite3.connect("analise_vendas.db")

# a query a ser executada
query = """ 
    SELECT
        strftime('%m', data_venda) AS mes,
        'R$ ' || format('%,.2f', SUM(quantidade * preco_unitario)) AS faturamento_total
    FROM vendas 
    GROUP BY mes
    ORDER BY mes
"""

# execução da query criada acima
df_resposta = pd.read_sql_query(query, con=conn)

# não esquecer de fechar a conexão
conn.close()

# exibir a resposta
df_resposta

,mes,faturamento_total
0,01,"R$ 1,332,505.24"
1,02,"R$ 1,414,699.44"
2,03,"R$ 1,561,420.22"
3,04,"R$ 1,571,459.42"
4,05,"R$ 1,537,043.09"
5,06,"R$ 1,346,568.09"
6,07,"R$ 1,528,937.90"
7,08,"R$ 1,531,974.02"
8,09,"R$ 1,560,269.23"
9,10,"R$ 1,423,821.79"


### 6. Quais são os 5 produtos mais vendidos?

In [31]:
# abre a conexão com o banco de dados
conn = sqlite3.connect("analise_vendas.db")

# a query a ser executada
query = """ 
    SELECT
        produto,
        SUM(quantidade) AS 'Quantidade vendida'
    FROM vendas 
    GROUP BY produto
    ORDER BY SUM(quantidade) DESC
"""

# execução da query criada acima
df_resposta = pd.read_sql_query(query, con=conn)

# não esquecer de fechar a conexão
conn.close()

# exibir a resposta
df_resposta.head()

,produto,Quantidade vendida
0,Monitor,2613
1,Mouse,2581
2,Mesa,2581
3,Teclado,2566
4,Mochila,2557


### 7. Qual cidade possui maior faturamento?

In [ ]:
# abre a conexão com o banco de dados
conn = sqlite3.connect("analise_vendas.db")

# a query a ser executada
query = """ 
    SELECT
        cidade,
        'R$ ' || format('%,.2f', SUM(quantidade * preco_unitario)) AS faturamento_total
    FROM vendas 
    GROUP BY cidade
    ORDER BY SUM(quantidade * preco_unitario) DESC
"""

# execução da query criada acima
df_resposta = pd.read_sql_query(query, con=conn)

# não esquecer de fechar a conexão
conn.close()

# exibir a resposta
print(f"A cidade que apresentou maior faturamento foi {df_resposta['cidade'][0]} com um total de {df_resposta['faturamento_total'][0]}")

A cidade que apresentou maior faturamento foi Curitiba com um total de R$ 3,636,297.53


### 8. Qual cliente comprou mais em valor?

In [36]:
# abre a conexão com o banco de dados
conn = sqlite3.connect("analise_vendas.db")

# a query a ser executada
query = """ 
    SELECT
        cliente,
        'R$ ' || format('%,.2f', SUM(quantidade * preco_unitario)) AS faturamento_total
    FROM vendas 
    GROUP BY cliente
    ORDER BY SUM(quantidade * preco_unitario) DESC
    LIMIT 1
"""

# execução da query criada acima
df_resposta = pd.read_sql_query(query, con=conn)

# não esquecer de fechar a conexão
conn.close()

# exibir a resposta
print(f"O cliente que mais comprou, em valor, foi {df_resposta['cliente'][0]} com um total de {df_resposta['faturamento_total'][0]}")

O cliente que mais comprou, em valor, foi Mathias com um total de R$ 122,641.69
